# Week 5 (starter): RAG Pipeline with Retrieval Evaluation

Retrieval runs fully local and free; only generation needs a key (set `GEMINI_API_KEY`, else it is skipped with a notice). Cells marked **TODO (you)** are yours. Dependencies: `sentence-transformers`, `faiss-cpu`. For generation: `pip install openai`.

In [1]:
import os, re, pathlib, numpy as np
os.environ['HF_HOME'] = str((pathlib.Path('.') / '.hf_cache').resolve())
from sentence_transformers import SentenceTransformer
import faiss
embedder = SentenceTransformer('sentence-transformers/all-MiniLM-L6-v2')

c:\Users\wyatt\Documents\Gradschool\cosc_650\repos\cosc-650-applied-llm-systems\src\week5\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 2700.81it/s]


## Part 1: Build the pipeline
Ingest and chunk a documentation corpus of at least eight documents, embed the chunks locally with sentence-transformers, and store them in a vector index (FAISS or similar). Implement retrieval that embeds a query and returns the top-k chunks.

In [2]:
from pypdf import PdfReader

DOCS_DIR = pathlib.Path("gardening_docs")

def load_pdf(file_path):
    reader = PdfReader(file_path)
    pages = []

    for page in reader.pages:
        text = page.extract_text()

        if text:
            pages.append(text)

    return "\n\n".join(pages)


documents = []

for file_path in DOCS_DIR.glob("*.pdf"):
    documents.append({
        "source": file_path.name,
        "text": load_pdf(file_path)
    })


print(f"Loaded {len(documents)} documents")

for doc in documents:
    print(f"{doc['source']}: {len(doc['text']):,} characters")

Loaded 8 documents
bush_gardening.pdf: 147,544 characters
fruit_gardening.pdf: 24,727 characters
gardening_beds.pdf: 20,128 characters
gardening_pests.pdf: 10,832 characters
gardening_planters.pdf: 34,943 characters
grass_gardening.pdf: 33,280 characters
tree_planting.pdf: 23,515 characters
vegetable_gardening.pdf: 78,865 characters


In [3]:
def chunk_fixed(text, size, overlap):
    text = re.sub(r'\s+', ' ', text).strip(); out, i = [], 0
    while i < len(text):
        out.append(text[i:i+size]); i += size - overlap
    return out

def chunk_paragraph(text):
    return [re.sub(r'\s+',' ',p).strip() for p in text.split('\n\n') if p.strip()]


def chunk_documents(documents, chunker):
    chunks = []

    for doc in documents:
        doc_chunks = chunker(doc["text"])

        for chunk_id, chunk in enumerate(doc_chunks):
            chunks.append({
                "source": doc["source"],
                "chunk_id": chunk_id,
                "text": chunk
            })

    return chunks


configs = {
    "small (120/20)": chunk_documents(
        documents,
        lambda text: chunk_fixed(text, 120, 20)
    ),
    "large (320/40)": chunk_documents(
        documents,
        lambda text: chunk_fixed(text, 320, 40)
    ),
    "by-paragraph": chunk_documents(
        documents,
        chunk_paragraph
    )
}

for n, c in configs.items():
    print(f'{n:16s} -> {len(c)} chunks')

def build(chunks):
    texts = [chunk["text"] for chunk in chunks]

    e = embedder.encode(
        texts,
        normalize_embeddings=True
    ).astype("float32")

    idx = faiss.IndexFlatIP(e.shape[1])
    idx.add(e)

    return idx

# def retrieve(idx, chunks, q, k=3):
#     qe = embedder.encode([q], normalize_embeddings=True).astype('float32')
#     # FAISS uses -1 for missing results when fewer than k chunks exist.
#     return [chunks[i] for i in idx.search(qe, k)[1][0] if i >= 0]

def retrieve(idx, chunks, q, k=3):
    qe = embedder.encode(
        [q],
        normalize_embeddings=True
    ).astype("float32")

    scores, indices = idx.search(qe, k)

    results = []

    for score, i in zip(scores[0], indices[0]):
        if i >= 0:
            results.append({
                "source": chunks[i]["source"],
                "chunk_id": chunks[i]["chunk_id"],
                "text": chunks[i]["text"],
                "score": float(score)
            })

    return results

small (120/20)   -> 3668 chunks
large (320/40)   -> 1313 chunks
by-paragraph     -> 200 chunks


In [4]:
indices = {
    name: build(chunks)
    for name, chunks in configs.items()
}

for name, idx in indices.items():
    print(f"{name:16s} -> {idx.ntotal} vectors")

small (120/20)   -> 3668 vectors
large (320/40)   -> 1313 vectors
by-paragraph     -> 200 vectors


In [5]:
query = "How much sunlight do vegetables need?"

results = retrieve(
    indices["by-paragraph"],
    configs["by-paragraph"],
    query,
    k=3
)

for i, result in enumerate(results, start=1):
    print(f"Result {i}")
    print(f"Source: {result['source']}")
    print(f"Chunk: {result['chunk_id']}")
    print(f"Score: {result['score']:.4f}")
    print(f"Text: {result['text']}")
    print()

Result 1
Source: vegetable_gardening.pdf
Chunk: 2
Score: 0.5475
Text: Vegetable Gardening 6-3 prune branches to allow in more light. If a solid fence is the source of the shade, consider replacing it with a post and rail fence that will let in more light. Light can be improved by painting nearby walls or fences white so they are more reflective. Areas receiving less sunlight may make a good composting site or work area. If the only garden site you have is shady, start by growing crops with low light requirements, such as leafy green crops like lettuce, or asparagus, garlic, and leeks. If these do well, expand into medium light-needing crops such as beans, radishes, and peas. Water. Regardless of where you live in Washington, if you are going to grow summer vegetables you will need a steady supply of clean water for irrigation all summer long. The easiest way to irrigate is from a water spigot located in or near your vegetable garden. If the spigot is in an inconvenient place, you may w

## Part 2: Evaluate retrieval
Compare at least two embedding models, or at least three chunking configurations, on at least ten test queries for which you decide in advance which retrieved chunks count as relevant. Report precision and recall at the chunk level, and explain where the choices disagree and why.

In [7]:
def find_chunks(config_name, source, keywords):
    chunks = configs[config_name]

    print (f"config: {config_name} source: {source} keywords: {keywords}")
    # Allow either one keyword or a list of keywords
    if isinstance(keywords, str):
        keywords = [keywords]

    matches = []

    for chunk in chunks:
        if chunk["source"] != source:
            continue

        text_lower = chunk["text"].lower()

        if any(keyword.lower() in text_lower for keyword in keywords):
            matches.append(chunk)

    for chunk in matches:
        print(f"Chunk {chunk['chunk_id']}")
        print(chunk["text"])
        print("-" * 80)

    print(f"Found {len(matches)} matching chunks.")

The below function call was ran for every combination of query and chunking configuration. Then results were input into chatgpt 5.6 Sol to determine which chunks were relevant.

In [20]:
# Query 10: How should a lawn be watered and maintained?

# find_chunks(
#     "small (120/20)",
#     "grass_gardening.pdf",
#     "water"
# )

# find_chunks(
#     "large (320/40)",
#     "grass_gardening.pdf",
#     "water"
# )

# find_chunks(
#     "by-paragraph",
#     "grass_gardening.pdf",
#     "water"
# )
find_chunks(
    "by-paragraph",
    "grass_gardening.pdf",
    [
        "water",
        "watering",
        "irrigation",
        "maintenance",
        "mowing",
        "fertilizer",
        "fertilizing"
    ]
)

config: by-paragraph source: grass_gardening.pdf keywords: ['water', 'watering', 'irrigation', 'maintenance', 'mowing', 'fertilizer', 'fertilizing']
Chunk 0
DIRECTORY WATER SUPPLY � � � � � � � � � � � � � � � � � � � � � � � � � � � � � � � � � � � � � � � � � 2 LOOKING AT LAWNS � � � � � � � � � � � � � � � � � � � � � � � � � � � � � � � � � � � � 3 BENEFITS OF LAWNS� � � � � � � � � � � � � � � � � � � � � � � � � � � � � � � � � � � � 3 PLANTING A NEW LAWN � � � � � � � � � � � � � � � � � � � � � � � � � � � � � � � � 4 What Type of Grass to Plant � � � � � � � � � � � � � � � � � � � � � � � � � � � � � 4 Perennial Ryegrass � � � � � � � � � � � � � � � � � � � � � � � � � � � � � � � � � � � � � � � 4 Tall Fescues � � � � � � � � � � � � � � � � � � � � � � � � � � � � � � � � � � � � � � � � � � � � 5 Fine Fescues � � � � � � � � � � � � � � � � � � � � � � � � � � � � � � � � � � � � � � � � � � � � 5 Bentgrass � � � � � � � � � � � � � � � � � � � � � � � � � � � � � � � � �

In [13]:
relevant_chunks = {
    "How much sunlight do vegetables need?": {
        "small (120/20)": {("vegetable_gardening.pdf", 28)},
        "large (320/40)": {("vegetable_gardening.pdf", 10)},
        "by-paragraph": {("vegetable_gardening.pdf", 1)}
    },

    "How should I choose a location for a vegetable garden?": {
        "small (120/20)": {("vegetable_gardening.pdf", 26),("vegetable_gardening.pdf", 27)},
        "large (320/40)": {("vegetable_gardening.pdf", 9)},
        "by-paragraph": {("vegetable_gardening.pdf", 1)}
    },

    "What type of soil is best for planting fruit trees?": {
        "small (120/20)": {("fruit_gardening.pdf", 12),("fruit_gardening.pdf", 13)},
        "large (320/40)": {("fruit_gardening.pdf", 4)},
        "by-paragraph": {("fruit_gardening.pdf", 0)}
    },

    "How should fruit trees be watered after planting?": {
        "small (120/20)": {("fruit_gardening.pdf", 39),("fruit_gardening.pdf", 46),("fruit_gardening.pdf", 47)},
        "large (320/40)": {("fruit_gardening.pdf", 14),("fruit_gardening.pdf", 16)},
        "by-paragraph": {("fruit_gardening.pdf", 0),("fruit_gardening.pdf", 1)}
    },

    "What are the benefits of using raised garden beds?": {
        "small (120/20)": {("gardening_beds.pdf", 26),
                            ("gardening_beds.pdf", 27),
                            ("gardening_beds.pdf", 28),
                            ("gardening_beds.pdf", 29),
                            ("gardening_beds.pdf", 30),
                            ("gardening_beds.pdf", 31),
                            ("gardening_beds.pdf", 32)},
        "large (320/40)": {("gardening_beds.pdf", 9),
                            ("gardening_beds.pdf", 10),
                            ("gardening_beds.pdf", 11),
                            ("gardening_beds.pdf", 13)},
        "by-paragraph": {("gardening_beds.pdf", 1)}
    },

    "How wide should a raised garden bed be?": {
        "small (120/20)": {("gardening_beds.pdf", 39)},
        "large (320/40)": {("gardening_beds.pdf", 14)},
        "by-paragraph": {("gardening_beds.pdf", 1)}
    },

    "How can crop rotation help control garden insects?": {
        "small (120/20)": {("gardening_pests.pdf", 22),
                            ("gardening_pests.pdf", 23),
                            ("gardening_pests.pdf", 24),
                            ("gardening_pests.pdf", 80)},
        "large (320/40)": {("gardening_pests.pdf", 7),("gardening_pests.pdf", 28)},
        "by-paragraph": {("gardening_pests.pdf", 0),("gardening_pests.pdf", 2)}
    },

    "How should shrubs be selected for different sunlight conditions?": {
        "small (120/20)": {("bush_gardening.pdf", 420),
                            ("bush_gardening.pdf", 421),
                            ("bush_gardening.pdf", 422),
                            ("bush_gardening.pdf", 568),
                            ("bush_gardening.pdf", 569),
                            ("bush_gardening.pdf", 570)},
        "large (320/40)": {("bush_gardening.pdf", 150),("bush_gardening.pdf", 203)},
        "by-paragraph": {("bush_gardening.pdf", 20),("bush_gardening.pdf", 27)}
    },

    "What should I consider when selecting a tree planting site?": {
        "small (120/20)": {("tree_planting.pdf", 15),
                            ("tree_planting.pdf", 16),
                            ("tree_planting.pdf", 17),
                            ("tree_planting.pdf", 18),
                            ("tree_planting.pdf", 19),
                            ("tree_planting.pdf", 20),
                            ("tree_planting.pdf", 21),
                            ("tree_planting.pdf", 27),
                            ("tree_planting.pdf", 28),
                            ("tree_planting.pdf", 29),
                            ("tree_planting.pdf", 30),
                            ("tree_planting.pdf", 31),
                            ("tree_planting.pdf", 32),
                            ("tree_planting.pdf", 33),
                            ("tree_planting.pdf", 34),
                            ("tree_planting.pdf", 35),
                            ("tree_planting.pdf", 36),
                            ("tree_planting.pdf", 37),
                            ("tree_planting.pdf", 38)},
        "large (320/40)": {("tree_planting.pdf", 5),
                            ("tree_planting.pdf", 6),
                            ("tree_planting.pdf", 10),
                            ("tree_planting.pdf", 12),
                            ("tree_planting.pdf", 13)},
        "by-paragraph": {("tree_planting.pdf", 1),("tree_planting.pdf", 2)}
    },

    "How should a lawn be watered and maintained?": {
        "small (120/20)": {("grass_gardening.pdf", 163),
                            ("grass_gardening.pdf", 164),
                            ("grass_gardening.pdf", 165),
                            ("grass_gardening.pdf", 166),
                            ("grass_gardening.pdf", 168),
                            ("grass_gardening.pdf", 169),
                            ("grass_gardening.pdf", 170),
                            ("grass_gardening.pdf", 171),
                            ("grass_gardening.pdf", 173),
                            ("grass_gardening.pdf", 174),
                            ("grass_gardening.pdf", 176),
                            ("grass_gardening.pdf", 177),
                            ("grass_gardening.pdf", 178),
                            ("grass_gardening.pdf", 179),
                            ("grass_gardening.pdf", 180),
                            ("grass_gardening.pdf", 181),
                            ("grass_gardening.pdf", 182),
                            ("grass_gardening.pdf", 183),
                            ("grass_gardening.pdf", 184),
                            ("grass_gardening.pdf", 185),
                            ("grass_gardening.pdf", 187),
                            ("grass_gardening.pdf", 189),
                            ("grass_gardening.pdf", 190),
                            ("grass_gardening.pdf", 191),
                            ("grass_gardening.pdf", 192),
                            ("grass_gardening.pdf", 200),
                            ("grass_gardening.pdf", 201),
                            ("grass_gardening.pdf", 204),
                            ("grass_gardening.pdf", 205),
                            ("grass_gardening.pdf", 206),
                            ("grass_gardening.pdf", 207),
                            ("grass_gardening.pdf", 208),
                            ("grass_gardening.pdf", 209),
                            ("grass_gardening.pdf", 210),
                            ("grass_gardening.pdf", 211),
                            ("grass_gardening.pdf", 216),
                            ("grass_gardening.pdf", 217),
                            ("grass_gardening.pdf", 218),
                            ("grass_gardening.pdf", 219),
                            ("grass_gardening.pdf", 220),
                            ("grass_gardening.pdf", 221),
                            ("grass_gardening.pdf", 230),
                            ("grass_gardening.pdf", 231),
                            ("grass_gardening.pdf", 232),
                            ("grass_gardening.pdf", 233),
                            ("grass_gardening.pdf", 237)},
        "large (320/40)": {("grass_gardening.pdf", 58),
                            ("grass_gardening.pdf", 59),
                            ("grass_gardening.pdf", 60),
                            ("grass_gardening.pdf", 61),
                            ("grass_gardening.pdf", 62),
                            ("grass_gardening.pdf", 63),
                            ("grass_gardening.pdf", 64),
                            ("grass_gardening.pdf", 65),
                            ("grass_gardening.pdf", 66),
                            ("grass_gardening.pdf", 67),
                            ("grass_gardening.pdf", 68),
                            ("grass_gardening.pdf", 71),
                            ("grass_gardening.pdf", 72),
                            ("grass_gardening.pdf", 73),
                            ("grass_gardening.pdf", 74),
                            ("grass_gardening.pdf", 77),
                            ("grass_gardening.pdf", 82),
                            ("grass_gardening.pdf", 83),
                            ("grass_gardening.pdf", 84)},
        "by-paragraph": {("grass_gardening.pdf", 9),
                        ("grass_gardening.pdf", 10),
                        ("grass_gardening.pdf", 11),
                        ("grass_gardening.pdf", 12),
                        ("grass_gardening.pdf", 13),
                        ("grass_gardening.pdf", 14),
                        ("grass_gardening.pdf", 15)}
    }
}

In [14]:
# TODO (you): your own queries and the fact each answer must contain.
test_queries = [
    "How much sunlight do vegetables need?",
    "How should I choose a location for a vegetable garden?",
    "What type of soil is best for planting fruit trees?",
    "How should fruit trees be watered after planting?",
    "What are the benefits of using raised garden beds?",
    "How wide should a raised garden bed be?",
    "How can crop rotation help control garden insects?",
    "How should shrubs be selected for different sunlight conditions?",
    "What should I consider when selecting a tree planting site?",
    "How should a lawn be watered and maintained?"
]
# This example checks whether retrieved context contains the answer.
# It does not calculate the chunk-level precision and recall required below.
# for n, c in configs.items():
#     hits = sum(1 for q, fact in qa if fact.lower() in ' '.join(retrieve(indices[n], c, q)).lower())
#     print(f'{n:16s} answer-recall@3: {hits}/{len(qa)} = {hits/len(qa):.2f}')
# TODO (you): compare at least two embedding models OR at least three chunking
# configurations on at least ten queries. Label relevant chunks in advance,
# report chunk-level precision and recall, and explain where the choices disagree.

In [15]:
def evaluate_query(config_name, query, k=3):
    chunks = configs[config_name]
    idx = indices[config_name]

    results = retrieve(idx, chunks, query, k=k)

    retrieved = {
        (result["source"], result["chunk_id"])
        for result in results
    }

    relevant = relevant_chunks[query][config_name]

    true_positives = retrieved & relevant
    
    precision = len(true_positives) / len(retrieved) if retrieved else 0.0
    recall = len(true_positives) / len(relevant) if relevant else 0.0

    return {
        "precision": precision,
        "recall": recall,
        "retrieved": retrieved,
        "relevant": relevant,
        "true_positives": true_positives
    }

In [16]:
import pandas as pd

evaluation_results = []

for query in test_queries:
    for config_name in configs:
        result = evaluate_query(
            config_name,
            query,
            k=3
        )

        evaluation_results.append({
            "query": query,
            "config": config_name,
            "precision@3": result["precision"],
            "recall@3": result["recall"],
            "hit@3": int(len(result["true_positives"]) > 0),
            "relevant_chunks": len(result["relevant"]),
            "true_positives": len(result["true_positives"])
        })

results_df = pd.DataFrame(evaluation_results)

results_df

,query,config,precision@3,recall@3,hit@3,relevant_chunks,true_positives
0,How much sunlight do vegetables need?,small (120/20),0.333333,1.000000,1,1,1
1,How much sunlight do vegetables need?,large (320/40),0.333333,1.000000,1,1,1
2,How much sunlight do vegetables need?,by-paragraph,0.333333,1.000000,1,1,1
3,How should I choose a location for a vegetable...,small (120/20),0.000000,0.000000,0,2,0
4,How should I choose a location for a vegetable...,large (320/40),0.333333,1.000000,1,1,1
5,How should I choose a location for a vegetable...,by-paragraph,0.333333,1.000000,1,1,1
6,What type of soil is best for planting fruit t...,small (120/20),0.333333,0.500000,1,2,1
7,What type of soil is best for planting fruit t...,large (320/40),0.000000,0.000000,0,1,0
8,What type of soil is best for planting fruit t...,by-paragraph,0.333333,1.000000,1,1,1
9,How should fruit trees be watered after planting?,small (120/20),0.000000,0.000000,0,3,0


In [17]:
summary_df = (
    results_df
    .groupby("config")
    .agg({
        "precision@3": "mean",
        "recall@3": "mean",
        "hit@3": "mean"
    })
    .round(3)
)

summary_df

,precision@3,recall@3,hit@3
config,,,
by-paragraph,0.433,0.779,0.9
large (320/40),0.200,0.420,0.5
small (120/20),0.100,0.167,0.3


The code below was used to verify top 3 chunks for queries. It requires some manual configuration for each query that needs a deeper dive.

In [13]:
query = "How should a lawn be watered and maintained?"
config_name = "small (120/20)"

result = evaluate_query(config_name, query, k=3)

retrieved_results = retrieve(
    indices[config_name],
    configs[config_name],
    query,
    k=3
)

for rank, chunk in enumerate(retrieved_results, start=1):
    chunk_key = (chunk["source"], chunk["chunk_id"])

    print(f"Rank {rank}")
    print(f"Source: {chunk['source']}")
    print(f"Chunk: {chunk['chunk_id']}")
    print(f"Score: {chunk['score']:.4f}")
    print(f"Relevant: {chunk_key in result['relevant']}")
    print(chunk["text"])
    print("-" * 80)

Rank 1
Source: grass_gardening.pdf
Chunk: 150
Score: 0.7824
Relevant: False
REQUIREMENTS FOR NEW LAWNS New plantings need to be watered lightly and more frequently than established grass� In dry w
--------------------------------------------------------------------------------
Rank 2
Source: grass_gardening.pdf
Chunk: 102
Score: 0.6911
Relevant: False
re being developed that, once established, require less upkeep and watering than standard varieties � The Turf Grass Wat
--------------------------------------------------------------------------------
Rank 3
Source: grass_gardening.pdf
Chunk: 298
Score: 0.6874
Relevant: False
 great lawn alternatives, offering much of a lawn’s neatness and uniformity but often with less water and maintenance re
--------------------------------------------------------------------------------


## Part 2 Analysis

### Retrieval Evaluation

The small (120/20), Large (320/40) and by-paragraph chunking strategies were tested using manually selected relevant chunks. The retriever returned the top 3 chunks and with this I was able to calculate the metrics below. Precision@3 measures how many of the top three retrieved chunks were actually relevant to the query, while Recall@3 measures how many of all the known relevant chunks were successfully found within those top three results. I added an additional metric hit@3 to show whether at least 1 relevant chunk was retrieved.

| Chunking Configuration | Precision@3 | Recall@3 | Hit@3 |
|------------------------|------------:|---------:|------:|
| Small (120/20)         | 0.100       | 0.167    | 0.300 |
| Large (320/40)         | 0.200       | 0.420    | 0.500 |
| By-paragraph           | 0.433       | 0.779    | 0.900 |


The table above shows the aggregated metrics for all three chunking strategies. We can see that by-paragraph performed the best for precision@3, recall@3 and hit@3. This strategy was able to retrieve at least 1 relevant chunk for 9 of the 10 test queries. 

### Differences Between Chunking Strategies

The results showed that the retrieval method performs poorly for the small chunk strategy. Looking at "How should a lawn be watered and maintained?" in particular, I observed 0s accross the board. Semantically similar chunks were often returned in the small strategy that did not included the desired context. Each matched word carries more weight when the chunks are smaller, and context is often left out of the chunk completely.

On the other hand, larger chunks contained more context. For the following test query "How should shrubs be selected for different sunlight conditions?", both large and paragraph strategies saw a recall@3 of 1.0 compared to .167 for small. We can also see a contradiction to this in the "How wide should a raised garden bed be?" test query, where the large strategy had a recall@3 of 1.0, while paragraph found no relevant chunks. This indicates that larger chunks do contain more context but can make detailed facts less prominent within the embeddings.

There is a clear limitation that recall suffers depending on the chunk size. For the following test query "What should I consider when selecting a tree planting site?" had 19 relevant small chunks, but only two relevant paragraphs. When you're using k=3, you will never retrieve all small chunks. So it's notable that recall does not only reflect performance but also the larger context of the chunks. 

## Part 3: Generate
Add a generation step that builds a grounded prompt from the retrieved chunks and answers with the Gemini free tier.

In [18]:
def gemini_chat(messages, model='gemini-3.5-flash-lite', **kw):
    if not os.environ.get('GEMINI_API_KEY'):
        return None
    from openai import OpenAI
    client = OpenAI(api_key=os.environ['GEMINI_API_KEY'], base_url='https://generativelanguage.googleapis.com/v1beta/openai/')
    return client.chat.completions.create(model=model, messages=messages, **kw).choices[0].message.content

def answer(q, config='by-paragraph'):
    ctx = retrieve(indices[config], configs[config], q, k=3)
    prompt = 'Answer only from the context.\nContext:\n- ' + '\n- '.join(ctx) + f'\nQuestion: {q}\nAnswer:'
    out = gemini_chat([{'role':'user','content':prompt}])
    return out if out is not None else '[API-BLOCKED] set GEMINI_API_KEY to generate; grounded prompt was built'

def answer(q, config="by-paragraph"):
    ctx = retrieve(
        indices[config],
        configs[config],
        q,
        k=3
    )

    context = "\n\n".join(
        f"[Source: {chunk['source']}, Chunk: {chunk['chunk_id']}]\n"
        f"{chunk['text']}"
        for chunk in ctx
    )

    prompt = f"""Answer the question using only the provided context.
If the answer cannot be found in the context, say that the provided context does not contain enough information.

Context:
{context}

Question: {q}

Answer:"""

    out = gemini_chat([
        {
            "role": "user",
            "content": prompt
        }
    ])

    return (
        out if out is not None
        else "[API-BLOCKED] set GEMINI_API_KEY to generate; grounded prompt was built"
    )

In [15]:
import time
# generation_results = []

# for query in test_queries:
#     for config_name in configs:
#         time.sleep(10)
#         response = answer(
#             query,
#             config=config_name
#         )

#         generation_results.append({
#             "query": query,
#             "config": config_name,
#             "response": response
#         })

generation_df = pd.DataFrame(generation_results)

pd.set_option("display.max_colwidth", None)
generation_df

NameError: name 'generation_results' is not defined

## Part 4: Find one failure and explain it
Show a query where retrieval surfaces the wrong chunk, or where a chunking choice splits a fact away from the words that would find it. Explain the cause and the mitigation.

In [ ]:
query = "How should a lawn be watered and maintained?"
config = "small (120/20)"

results = retrieve(
    indices[config],
    configs[config],
    query,
    k=3
)

for rank, chunk in enumerate(results, start=1):
    print(f"Rank {rank}")
    print(f"Source: {chunk['source']}")
    print(f"Chunk: {chunk['chunk_id']}")
    print(f"Score: {chunk['score']:.4f}")
    print(f"Text: {chunk['text']}")
    print("-" * 80)

query = "How should a lawn be watered and maintained?"

response = answer(
    query,
    config="small (120/20)"
)

print(response)

Rank 1
Source: grass_gardening.pdf
Chunk: 150
Score: 0.7824
Text: REQUIREMENTS FOR NEW LAWNS New plantings need to be watered lightly and more frequently than established grass� In dry w
--------------------------------------------------------------------------------
Rank 2
Source: grass_gardening.pdf
Chunk: 102
Score: 0.6911
Text: re being developed that, once established, require less upkeep and watering than standard varieties � The Turf Grass Wat
--------------------------------------------------------------------------------
Rank 3
Source: grass_gardening.pdf
Chunk: 298
Score: 0.6874
Text:  great lawn alternatives, offering much of a lawn’s neatness and uniformity but often with less water and maintenance re
--------------------------------------------------------------------------------
Based on the provided context, new plantings need to be watered lightly and more frequently than established grass. Additionally, certain grass varieties and lawn alternatives are being developed 

## Part 4 analysis

### Failure Analysis
For the following test query: "How should a lawn be watered and maintained?", the small chunking strategy failed to retrieve directly relevant chunks. The retrieved chunks discuss new lawns, low maintenance grass varieties and lawn alternatives. This resulted in Gimini providing a partial response and following our prompt to respond accordingly when context is insufficient.

As noted earlier, the small chunking strategy tends to leave out important context and prioritize semantic similarity. The retrieved chunks contained similar words such as lawn, water and maintenance, but did not contain context needed to answer the question.

We can mitigate this by using larger or paragraph chunk strategies to preserve context and improve retrieval. In this project, paragraph chunking was able to retrieve relevant information for this failure query. If you're intent on improve the small strategy performance, you could also increase overlap between small chunks to help retain realted information from being segmented. 

## Week 6 Discussion

In [16]:
queries = [
    "What factors should I consider when choosing a location for my vegetable garden?",
    "How can I improve the soil in my garden before planting vegetables?",
    "How often should I water my vegetable garden, and what is the best way to water it?",
    "What are the benefits of using compost in a vegetable garden?",
    "How can I prevent and manage pests in my vegetable garden?"
]

config = "by-paragraph"

for i, query in enumerate(queries, start=1):
    print(f"\nQUERY {i}")
    print(f"Question: {query}")
    print("=" * 80)

    # Retrieve chunks
    results = retrieve(
        indices[config],
        configs[config],
        query,
        k=3
    )

    print("\nRETRIEVED CONTEXT")

    for rank, chunk in enumerate(results, start=1):
        print(f"\nRank {rank}")
        print(f"Source: {chunk['source']}")
        print(f"Chunk: {chunk['chunk_id']}")
        print(f"Score: {chunk['score']:.4f}")
        print(f"Text: {chunk['text']}")

    # Generate answer
    response = answer(
        query,
        config=config
    )

    print("\nGENERATED ANSWER")
    print(response)

    print("\n" + "-" * 80)




QUERY 1
Question: What factors should I consider when choosing a location for my vegetable garden?

RETRIEVED CONTEXT

Rank 1
Source: vegetable_gardening.pdf
Chunk: 1
Score: 0.6583
Text: 6-2 Vegetable Gardening Introduction Sustainable gardening means growing plants in an environmentally friendly way. In the vegetable garden, this may include using cover crops to improve soil tilth and nutrition, rotating crops to prevent disease, and choosing an irrigation system that conserves water and reduces disease pressure. Also, making the right choices regarding types of crops and varieties to grow will enable you to produce a healthy crop while reducing the need for pesticides. Many of the practices that we will discuss in this chapter are the same practices that are used in organic gardening. Our goal is to grow a healthy, productive vegetable garden while minimizing outside inputs such as purchased fertilizer and pesticides. In addition, by recycling plant debris through composting you wil

In [19]:
import sys
import langchain_community

print(sys.executable)
print(langchain_community.__version__)
import ragas

print(ragas.__version__)
from ragas.metrics import (
    Faithfulness,
    ResponseRelevancy,
    LLMContextPrecisionWithReference,
    LLMContextRecall,
)

c:\Users\wyatt\Documents\Gradschool\cosc_650\repos\cosc-650-applied-llm-systems\src\week5\.venv\Scripts\python.exe
0.3.31
0.3.9


In [20]:
import os

from langchain_openai import ChatOpenAI
from ragas.llms import LangchainLLMWrapper

gemini_evaluator = ChatOpenAI(
    model="gemini-3.5-flash-lite",
    api_key=os.environ["GEMINI_API_KEY"],
    base_url="https://generativelanguage.googleapis.com/v1beta/openai/",
    temperature=0,
)

evaluator_llm = LangchainLLMWrapper(gemini_evaluator)

C:\Users\wyatt\AppData\Local\Temp\ipykernel_18348\1294215970.py:13: DeprecationWarning: LangchainLLMWrapper is deprecated and will be removed in a future version. Use llm_factory instead: from openai import OpenAI; from ragas.llms import llm_factory; llm = llm_factory('gpt-4o-mini', client=OpenAI(api_key='...'))
  evaluator_llm = LangchainLLMWrapper(gemini_evaluator)


In [21]:
test = gemini_evaluator.invoke("Reply with only: working")

print(test.content)

working


In [22]:
queries = [
    "What factors should I consider when choosing a location for my vegetable garden?",
    "How can I improve the soil in my garden before planting vegetables?",
    "How often should I water my vegetable garden, and what is the best way to water it?",
    "What are the benefits of using compost in a vegetable garden?",
    "How can I prevent and manage pests in my vegetable garden?"
]

references = [
    """
    Choose a vegetable garden site with full sunlight, easy access to water,
    protection from heavy winds, and soil with good tilth. Most summer
    vegetable crops need at least eight hours of direct sunlight per day.
    """,

    """
    Improve garden soil by regularly adding organic matter such as compost.
    Add 2–3 inches of compost to beds or rows each year. Cover crops and
    composted animal manure can also improve soil. For rocky or compacted
    soil, build the soil gradually and consider deep-rooted vegetables to
    improve drainage and aeration.
    """,

    """
    Seedlings and transplants should be watered every 2–3 days for the first
    week or two. Afterward, irrigate about once per week or as needed,
    providing approximately one inch of water per week. Soaker hoses and
    drip irrigation are more efficient than sprinklers because they reduce
    evaporation, runoff, weed growth, and problems associated with wet foliage.
    """,

    """
    Compost adds organic matter and nutrients to garden soil. Organic matter
    improves soil health and helps feed plants. Composting also allows garden
    debris and kitchen scraps to be recycled, reducing the amount of waste
    produced.
    """,

    """
    Prevent and manage garden pests by maintaining healthy plants, properly
    spacing plants, keeping the garden clean, identifying pests before
    treatment, scouting regularly, rotating crops, and using beneficial
    insects when appropriate. Removing weeds and old plant debris can also
    reduce places where pests live.
    """
]

In [23]:
evaluation_data = []

config = "by-paragraph"

for query, reference in zip(queries, references):

    results = retrieve(
        indices[config],
        configs[config],
        query,
        k=3
    )

    contexts = [
        chunk["text"]
        for chunk in results
    ]

    response = answer(
        query,
        config=config
    )

    evaluation_data.append({
        "user_input": query,
        "retrieved_contexts": contexts,
        "response": response,
        "reference": reference
    })

In [24]:
evaluation_data[0]

{'user_input': 'What factors should I consider when choosing a location for my vegetable garden?',
 'retrieved_contexts': ['6-2 Vegetable Gardening Introduction Sustainable gardening means growing plants in an environmentally friendly way. In the vegetable garden, this may include using cover crops to improve soil tilth and nutrition, rotating crops to prevent disease, and choosing an irrigation system that conserves water and reduces disease pressure. Also, making the right choices regarding types of crops and varieties to grow will enable you to produce a healthy crop while reducing the need for pesticides. Many of the practices that we will discuss in this chapter are the same practices that are used in organic gardening. Our goal is to grow a healthy, productive vegetable garden while minimizing outside inputs such as purchased fertilizer and pesticides. In addition, by recycling plant debris through composting you will be able to reduce the amount of waste that your garden produce

In [25]:
from ragas.dataset_schema import SingleTurnSample

samples = [
    SingleTurnSample(
        user_input=item["user_input"],
        response=item["response"],
        retrieved_contexts=item["retrieved_contexts"],
        reference=item["reference"],
    )
    for item in evaluation_data
]

In [26]:
print(samples[0])

user_input='What factors should I consider when choosing a location for my vegetable garden?' retrieved_contexts=['6-2 Vegetable Gardening Introduction Sustainable gardening means growing plants in an environmentally friendly way. In the vegetable garden, this may include using cover crops to improve soil tilth and nutrition, rotating crops to prevent disease, and choosing an irrigation system that conserves water and reduces disease pressure. Also, making the right choices regarding types of crops and varieties to grow will enable you to produce a healthy crop while reducing the need for pesticides. Many of the practices that we will discuss in this chapter are the same practices that are used in organic gardening. Our goal is to grow a healthy, productive vegetable garden while minimizing outside inputs such as purchased fertilizer and pesticides. In addition, by recycling plant debris through composting you will be able to reduce the amount of waste that your garden produces. In thi

In [28]:
from langchain_core.embeddings import Embeddings

class SentenceTransformerAdapter(Embeddings):
    def __init__(self, embedder):
        self.embedder = embedder

    def embed_documents(self, texts):
        return self.embedder.encode(
            texts,
            normalize_embeddings=True
        ).tolist()

    def embed_query(self, text):
        return self.embedder.encode(
            text,
            normalize_embeddings=True
        ).tolist()


ragas_embeddings = SentenceTransformerAdapter(embedder)

In [32]:
from ragas.metrics import (
    Faithfulness,
    ResponseRelevancy,
    LLMContextPrecisionWithReference,
    LLMContextRecall,
)

faithfulness = Faithfulness(
    llm=evaluator_llm
)

answer_relevance = ResponseRelevancy(
    llm=evaluator_llm,
    embeddings=ragas_embeddings,
    strictness=1
)

context_precision = LLMContextPrecisionWithReference(
    llm=evaluator_llm
)

context_recall = LLMContextRecall(
    llm=evaluator_llm
)

In [33]:
score = await faithfulness.single_turn_ascore(samples[0])

print(f"Faithfulness: {score:.4f}")

Faithfulness: 1.0000


In [34]:
import pandas as pd

scores = []

for i, sample in enumerate(samples, start=1):
    print(f"Evaluating Query {i}...")

    faithfulness_score = await faithfulness.single_turn_ascore(sample)
    relevance_score = await answer_relevance.single_turn_ascore(sample)
    precision_score = await context_precision.single_turn_ascore(sample)
    recall_score = await context_recall.single_turn_ascore(sample)

    scores.append({
        "Query": i,
        "Faithfulness": faithfulness_score,
        "Answer Relevance": relevance_score,
        "Context Precision": precision_score,
        "Context Recall": recall_score
    })

results_df = pd.DataFrame(scores)

results_df

Evaluating Query 1...
Evaluating Query 2...
Evaluating Query 3...
Evaluating Query 4...
Evaluating Query 5...


,Query,Faithfulness,Answer Relevance,Context Precision,Context Recall
0,1,1.0,0.579308,1.000000,1.0
1,2,1.0,0.689938,0.833333,1.0
2,3,1.0,0.670021,1.000000,1.0
3,4,1.0,1.000000,0.500000,1.0
4,5,1.0,1.000000,0.833333,1.0


In [35]:
metric_averages = results_df[
    [
        "Faithfulness",
        "Answer Relevance",
        "Context Precision",
        "Context Recall"
    ]
].mean()

print(metric_averages)

Faithfulness         1.000000
Answer Relevance     0.787853
Context Precision    0.833333
Context Recall       1.000000
dtype: float64


In [36]:
weakest_dimension = metric_averages.idxmin()
weakest_score = metric_averages.min()

print(f"Weakest dimension: {weakest_dimension}")
print(f"Average score: {weakest_score:.4f}")

Weakest dimension: Answer Relevance
Average score: 0.7879


## Part 5: Review a classmate’s pull request
Review a classmate’s Week 5 pull request. Give specific feedback on their RAG pipeline or retrieval evaluation, and include the review URL with your Canvas submission.

Project review: https://github.com/fdx-hw/cosc-650/pull/17#issuecomment-5861215387